# Import packages

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
import matplotlib.pyplot as plt
import numpy as np

from typing import Dict, List, Tuple, Sequence


# Import raw weather data from Open-Meteo API

In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2020-10-15",
	"end_date": "2025-10-15",
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "rain", "wind_speed_10m", "cloud_cover", "pressure_msl"],
	"timezone": "Europe/Berlin",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}?N {response.Longitude()}?E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_rain = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(6).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(7).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["rain"] = hourly_rain
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["pressure_msl"] = hourly_pressure_msl

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

In [ ]:
#hourly_dataframe.to_csv("./raw_data/weather_data_20221015-20251015.csv")

# Data exploration and additional variables

In [ ]:
hourly_dataframe.shape

In [ ]:
hourly_dataframe.dtypes

In [ ]:
#Check basic statistics
hourly_dataframe.describe()

In [ ]:
#check null values
hourly_dataframe.isnull().sum()

In [ ]:
hourly_dataframe.set_index('date',inplace = True, drop = True)

In [ ]:
hourly_dataframe.head()

In [ ]:
#Plot every variable to check if we have obvious outliers
fig, ax = plt.subplots(2,4, figsize = (15,4))
axes = ax.flatten()

for i, a in enumerate(axes):
    a.plot(hourly_dataframe.iloc[:,i])
    a.set_title(f"{hourly_dataframe.columns[i]}")

plt.tight_layout()
plt.show()

In [ ]:
# Check if we have missing rows

# nb of rows expected
expected_rows= int((hourly_dataframe.index.max(
) - hourly_dataframe.index.min()).total_seconds() / 3600) +1
print(f'Expected number of rows : {expected_rows}')

# Current rows
actual_rows = len(hourly_dataframe)
print(f'Actual number of rows : {actual_rows}')

#Missing rows
missing_rows = expected_rows - actual_rows
print(f'Missing rows : {missing_rows}')


In [ ]:
def fill_missing_timestamps(df:pd.DataFrame)->pd.DataFrame:
    '''
    Check if some rows are missing; fill them with linear interpolation and return
    the full dataframe.
    Datetime data have to be the index.
    '''
    full_index = pd.date_range(start=df.index.min(),
                           end=df.index.max(),
                           freq='h')

    missing_timestamps = full_index.difference(df.index)

    df = df.reindex(full_index)
    df= df.interpolate(method='linear')
    df.index.name = 'datetime'
    if len(missing_timestamps) != 0:
        print(f"{len(missing_timestamps)} missing timestamp(s) interpolated")
    return df


In [ ]:
hourly_dataframe = fill_missing_timestamps(hourly_dataframe)

In [ ]:
#Retrieve data as a variable
hourly_dataframe = hourly_dataframe.reset_index().rename(columns={'index': 'date'})

In [ ]:
#Create cyclical variables
#NB : if we finally decide to take more years in the dataset, then we should compute cyclical variables
#for the year
hourly_dataframe['Day_of_Year_sin'] = np.sin(2 * np.pi * hourly_dataframe['date'].dt.dayofyear / 365)
hourly_dataframe['Day_of_Year_cos'] = np.cos(2 * np.pi * hourly_dataframe['date'].dt.dayofyear / 365)
hourly_dataframe['Day_of_Week_sin'] = np.sin(2 * np.pi * hourly_dataframe['date'].dt.weekday / 7)
hourly_dataframe['Day_of_Week_cos'] = np.cos(2 * np.pi * hourly_dataframe['date'].dt.weekday / 7)
hourly_dataframe['Hour_sin'] = np.sin(2 * np.pi * hourly_dataframe['date'].dt.hour / 24)
hourly_dataframe['Hour_cos'] = np.cos(2 * np.pi * hourly_dataframe['date'].dt.hour / 24)

In [ ]:
hourly_dataframe.head()

In [ ]:
#We don't need the date column anymore, so let's drop it
hourly_dataframe.drop(columns = ['date'], inplace = True)

In [ ]:
hourly_dataframe

?? In the preprocessed dataset, we have :
- $26$k rows  (~ 3 years of weather data)
- $14$ features composed of:
    - $4$ <font color=green>**target**</font> (we will use the past values of the temperature, wind, rain, humidity as a feature)
    - $10$ <font color=orange>**past covariates**</font> (= features which past values are known)
    - $0$ <font color=blue>**future covariates**</font> (= features which future values are known, e.g. public holidays)

In [ ]:
hourly_dataframe.shape

??????? This weather dataset is a DataFrame (dimension = 2) which is a single Time Series from the beginning of October 2022 to October 2025 with records every hour.

* `df.shape = (n_timesteps, n_features) = (26328, 14)`

?? The goal is to predict the temperature, wind, rain, humidity for the next 12 hours using the past values.

In [ ]:
# Let's define the global variables of our dataset
TARGET = ['temperature_2m','relative_humidity_2m', 'rain','wind_speed_10m']
N_TARGETS = 4
N_FEATURES = 14


In [ ]:
TARGET_INDEX = [hourly_dataframe.columns.get_loc(t) for t in TARGET]

# Data preparation

## Create folds for cross-validation

Each of them with shape `(FOLD_LENGTH, n_features)` 

Let's define some global variables that we will use for our tests everywhere in this notebook:

In [ ]:
# --------------------------------------------------- #
# Let's consider FOLDS with a length of 3 years       #
# (2 years will be used for train, 1 for test!)       #
# --------------------------------------------------- #

FOLD_LENGTH = 24*365 * 3 # every hour
                        # three years

# --------------------------------------------------- #
# Let's consider FOLDS starting every trimester       #
# --------------------------------------------------- #

FOLD_STRIDE = 24*91 # every hour
                   # 1 quarter = 91 days

# --------------------------------------------------- #
# Let's consider a train-test-split ratio of 2/3      #
# --------------------------------------------------- #

TRAIN_TEST_RATIO = 0.66


In [ ]:
def get_folds(
    df: pd.DataFrame,
    fold_length: int,
    fold_stride: int) -> List[pd.DataFrame]:
    """
    This function slides through the Time Series dataframe of shape (n_timesteps, n_features) to create folds
    - of equal `fold_length`
    - using `fold_stride` between each fold

    Args:
        df (pd.DataFrame): Overall dataframe
        fold_length (int): How long each fold should be in rows
        fold_stride (int): How many timesteps to move forward between taking each fold

    Returns:
        List[pd.DataFrame]: A list where each fold is a dataframe within
    """
    result = []
    i = 0
    while(fold_length   <= (df.shape[0] - i + 1)):
        tmp_df = df.iloc[i:(i+fold_length), :]
        result.append(tmp_df)
        i = i + fold_stride
    return result


In [ ]:
folds = get_folds(hourly_dataframe, FOLD_LENGTH, FOLD_STRIDE)

print(f'The function generated {len(folds)} folds.')
print(f'Each fold has a shape equal to {folds[0].shape}.')


## Temporal Train/Test split

In [ ]:
INPUT_DAYS = 3 #number of days for the input_length ; 3 days seems okay since we want to predict the next 12 hours
INPUT_LENGTH = 24 * INPUT_DAYS # records every hour
INPUT_LENGTH = 12 # records every hour
INPUT_LENGTH = 24*14
INPUT_LENGTH = 24

In [ ]:
def train_test_split(fold:pd.DataFrame,
                     train_test_ratio: float,
                     input_length: int) -> Tuple[pd.DataFrame]:
    """From a fold dataframe, take a train dataframe and test dataframe based on
    the split ratio.
    - df_train should contain all the timesteps until round(train_test_ratio * len(fold))
    - df_test should contain all the timesteps needed to create all (X_test, y_test) tuples

    Args:
        fold (pd.DataFrame): A fold of timesteps
        train_test_ratio (float): The ratio between train and test 0-1
        input_length (int): How long each X_i will be

    Returns:
        Tuple[pd.DataFrame]: A tuple of two dataframes (fold_train, fold_test)
    """

    fold_train = fold.iloc[0:round(train_test_ratio * len(fold)),:]
    fold_test = fold.iloc[round((train_test_ratio * len(fold)) - input_length):,:]

    return (fold_train, fold_test)


Let's test the function on one fold

In [ ]:
fold = folds[0]
(fold_train, fold_test) = train_test_split(fold, TRAIN_TEST_RATIO, INPUT_LENGTH)

In [ ]:
assert fold_train.shape[0] == round(TRAIN_TEST_RATIO * fold.shape[0])

## Create (X, y) sequences

In [ ]:
# INPUT X
print(f'N_FEATURES = {N_FEATURES}')
print(f'INPUT_LENGTH = {INPUT_LENGTH} timesteps = {int(INPUT_LENGTH/24)} days = {int(INPUT_LENGTH/24/7)} weeks')


?? `X` is the **input** of our model. 
- It contains $14$ features: the past values of the <font color=green>**target**</font> + $10$ <font color=orange>**past-covariates**</font>
- Each sequence has a length equal to $72$ timesteps (=$3$ days)

In [ ]:
# TARGET Y
print(f'N_TARGETS = {N_TARGETS}')

# Let's predict 12 values ahead of us
# OUTPUT_LENGTH = 12
OUTPUT_LENGTH = 24
print(f'OUTPUT_LENGTH = {OUTPUT_LENGTH}')

?? `y` is the <font color=green>**target**</font> that we want to predic:t
- It is the value at the 12 next timesteps (= 12 next hours)

Let's build sequences by scanning folds chronologically and create all the possible pairs $(X_i, y_i)$, based on : 
  
  - an `input_length` (let's still use `INPUT_LENGTH = 24 * 3`, i.e. two weeks) 
  - and a `sequence_stride`

?? Let's scan the fold with a temporal stride of 1 hours (the minimum one):

In [ ]:
SEQUENCE_STRIDE = 1  #We need to brainstorm about it   i had 3 before

In [ ]:
def get_X_y_strides(fold: pd.DataFrame, input_length: int, output_length: int,
    sequence_stride: int) -> Tuple[np.array]:
    """slides through a `fold` Time Series (2D array) to create sequences of equal
        * `input_length` for X,
        * `output_length` for y,
    using a temporal gap `sequence_stride` between each sequence

    Args:
        fold (pd.DataFrame): One single fold dataframe
        input_length (int): Length of each X_i
        output_length (int): Length of each y_i
        sequence_stride (int): How many timesteps to take before taking the next X_i

    Returns:
        Tuple[np.array]: A tuple of numpy arrays (X, y)
    """
    X, y = [], []

    for i in range(0, len(fold), sequence_stride):
        # Exits the loop as soon as the last fold index would exceed the last index
        if (i + input_length + output_length) >= len(fold):
            break
        X_i = fold.iloc[i:i + input_length, :]
        y_i = fold.iloc[i + input_length:i + input_length + output_length, :][TARGET]
        X.append(X_i)
        y.append(y_i)

    return (np.array(X), np.array(y))


??????? Some clarifications about scanning a fold sequentially :

In [ ]:
print("FOLD_LENGTH")
print(f"= {FOLD_LENGTH} timesteps")
print(f"= {int(FOLD_LENGTH/24)} days") # 24 records per day, every 1 hour
print(f"= {int(FOLD_LENGTH/24/7)} weeks")

In [ ]:
#Build the X_train,y_train, X_test, y_test for one fold

X_train, y_train = get_X_y_strides(fold_train, INPUT_LENGTH, OUTPUT_LENGTH, SEQUENCE_STRIDE)
X_test, y_test = get_X_y_strides(fold_test, INPUT_LENGTH, OUTPUT_LENGTH, SEQUENCE_STRIDE)

print(X_train.shape)
print(y_train.shape)

In [ ]:
print(X_test.shape)
print(y_test.shape)

In [ ]:
N_TRAIN = X_train.shape[0]
N_TEST = X_test.shape[0]

In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers, optimizers

def init_baseline():
    forecast_horizon= OUTPUT_LENGTH
    target_indices = TARGET_INDEX
    model = models.Sequential()
    model.add(layers.Lambda(lambda x: tf.tile(
        tf.expand_dims(tf.gather(x[:, -1, :], target_indices, axis=1), axis=1),
        [1, forecast_horizon, 1]
    )))
    model.compile(loss='mse', optimizer=optimizers.Adam(0.02), metrics=['mae'])
    return model


baseline_model = init_baseline()

y_pred_baseline = baseline_model.predict(X_test)
print("y_pred_baseline shape:", y_pred_baseline.shape)

loss, mae = baseline_model.evaluate(X_test, y_test)
print(f"Baseline MSE: {loss:.4f}, MAE: {mae:.4f}")


# Modelling

**The MAE as a metrics to monitor the temperature prediction**

## Definition of the model and evaluation on a single fold

In [ ]:
from tensorflow import keras

In [ ]:
from tensorflow import keras
from tensorflow.keras import models, layers, optimizers, metrics
from tensorflow.keras.regularizers import L1L2
from tensorflow.keras.layers import Normalization, Input

def init_model(X_train, y_train):

    # 0 - Normalization
    # ======================
    normalizer = Normalization()
    normalizer.adapt(X_train)

    # 1 - RNN architecture
    # ======================
    model = models.Sequential()
    #model.add(Input(shape=X_train[0].shape)) --> no need because Normalization apprend l'input
    ## 1.0 - All the rows will be standardized through the already adapted normalization layer
    model.add(normalizer)
    ## 1.1 - Recurrent Layer
    model.add(layers.LSTM(64,
                          activation='tanh',
                          return_sequences = True,
                          kernel_regularizer=L1L2(l1=0.01, l2=0.01),
                          ))

    ## 1.2 - Predictive Dense Layers
    output_length = y_train.shape[2]
    model.add(layers.TimeDistributed(layers.Dense(output_length, activation='linear')))
    #model.add(layers.Dense(output_length, activation='linear'))

    # 2 - Compiler
    # ======================
    adam = optimizers.Adam(learning_rate=0.001)
    model.compile(loss='mse', optimizer=adam, metrics=["mae"])

    return model

In [ ]:
X_train.shape[1:]

In [ ]:
#Other version

from tensorflow import keras
from tensorflow.keras import models, layers, optimizers, metrics,Sequential
from tensorflow.keras.regularizers import L1L2
from tensorflow.keras.layers import Normalization, Input,Bidirectional,LSTM

def init_model(X_train, y_train):

    model = Sequential()
    normalizer = Normalization()
    normalizer.adapt(X_train)
    model.add(layers.Bidirectional(LSTM(256, return_sequences=True), input_shape=X_train.shape[1:]))
    # model.add(LayerNormalization())
    # model.add(Dropout(0.2))
    model.add(layers.Bidirectional(LSTM(128, return_sequences=True)))
    # model.add(LayerNormalization())
    # model.add(Dropout(0.2))
    model.add(Bidirectional(LSTM(64, return_sequences=True)))
    # model.add(Dropout(0.2))
    model.add(layers.Dense(64, activation='relu'))
    output_length = y_train.shape[2]
    model.add(layers.Dense(output_length, activation='linear'))

    # 2 - Compiler
    # ======================
    adam = optimizers.Adam(learning_rate=0.001)
    model.compile(loss='mse', optimizer=adam, metrics=["mae"])

    return model

In [ ]:
y_train.shape

In [ ]:
model = init_model(X_train, y_train)
model.summary()

In [ ]:
def plot_history(history):

    fig, ax = plt.subplots(1,2, figsize=(20,7))
    # --- LOSS: MSE ---
    ax[0].plot(history.history['loss'])
    ax[0].plot(history.history['val_loss'])
    ax[0].set_title('MSE')
    ax[0].set_ylabel('Loss')
    ax[0].set_xlabel('Epoch')
    ax[0].legend(['Train', 'Validation'], loc='best')
    ax[0].grid(axis="x",linewidth=0.5)
    ax[0].grid(axis="y",linewidth=0.5)

    # --- METRICS:MAE ---

    ax[1].plot(history.history['mae'])
    ax[1].plot(history.history['val_mae'])
    ax[1].set_title('MAE')
    ax[1].set_ylabel('MAE')
    ax[1].set_xlabel('Epoch')
    ax[1].legend(['Train', 'Validation'], loc='best')
    ax[1].grid(axis="x",linewidth=0.5)
    ax[1].grid(axis="y",linewidth=0.5)

    return ax


In [ ]:
from keras.callbacks import EarlyStopping


def fit_model(model):
    es = EarlyStopping(patience = 3,monitor= 'val_loss', restore_best_weights = True, mode = 'min')

    history = model.fit(X_train,
                        y_train,
                        epochs = 50,
                        batch_size = 32,
                        validation_split= 0.3,
                        callbacks = [es],
                        verbose = 0
    )

    return model,history

In [ ]:
# 1 - Initializing the RNN model
# ====================================

model = init_model(X_train, y_train)
model.summary()

# 2 - Training
# ====================================
model, history = fit_model(model)

In [ ]:
plot_history(history);

In [ ]:
# 3 - Evaluation
# ====================================
res = model.evaluate(X_test, y_test)
print(f"The LSTM MAE on the test set is equal to {round(res[1],2)} Celsius degrees")


In [ ]:
y_pred = model.predict(X_test)
y_pred.shape

In [ ]:
y_pred[0,:,0]

In [ ]:
y_test[0,:,0]

## Cross-validation

In [ ]:
# Reminders of the global variables in this notebook --> TO DELETE

print(f'N_FEATURES = {N_FEATURES}')
print(f'N_TARGETS = {N_TARGETS}')
print('')
print(f'FOLD_LENGTH = {FOLD_LENGTH}')
print(f'FOLD_STRIDE = {FOLD_STRIDE}')
print(f'TRAIN_TEST_RATIO = {TRAIN_TEST_RATIO}')
print('')
print(f'N_TRAIN = {N_TRAIN}')
print(f'N_TEST = {N_TEST}')
print(f'INPUT_LENGTH = {INPUT_LENGTH}')
print(f'OUTPUT_LENGTH = {OUTPUT_LENGTH}')

In [ ]:
from keras.callbacks import EarlyStopping

def cross_validate_lstm():
    '''
    This function cross-validates
    - the "last seen value" baseline model
    - the RNN model
    '''

    list_of_mae_baseline_model = []
    list_of_mae_recurrent_model = []

    # 0 - Creating folds
    # =========================================
    folds = get_folds(hourly_dataframe, FOLD_LENGTH, FOLD_STRIDE)

    for fold_id, fold in enumerate(folds):

        # 1 - Train/Test split the current fold
        # =========================================
        (fold_train, fold_test) = train_test_split(fold, TRAIN_TEST_RATIO, INPUT_LENGTH)

        X_train, y_train = get_X_y_strides(fold_train, INPUT_LENGTH, OUTPUT_LENGTH, SEQUENCE_STRIDE)
        X_test, y_test = get_X_y_strides(fold_test, INPUT_LENGTH, OUTPUT_LENGTH, SEQUENCE_STRIDE)

        # 2 - Modelling
        # =========================================

        ##### LSTM Model
        model = init_model(X_train, y_train)
        es = EarlyStopping(monitor = "val_mae",
                           mode = "min",
                           patience = 2,
                           restore_best_weights = True)
        history = model.fit(X_train, y_train,
                            validation_split = 0.3,
                            shuffle = False,
                            batch_size = 32,
                            epochs = 50,
                            callbacks = [es],
                            verbose = 0)
        res = model.evaluate(X_test, y_test, verbose=0)
        mae_lstm = res[1]
        list_of_mae_recurrent_model.append(mae_lstm)
        print(f"MAE LSTM fold n?{fold_id} = {round(mae_lstm, 2)}")

    return list_of_mae_recurrent_model


In [ ]:
mae_lstms = cross_validate_lstm()